In [ ]:
# Import libraries 
import pandas as pd
import requests
import json
import time
import datetime
import uuid
import plotly.express as px
from dotenv import load_dotenv

In [ ]:
# Load .env vars 
load_dotenv()
weather_api_secret = os.getenv('WEATHER_API_SECRET')
aws_access_key_id = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')
bucket_name = os.getenv('BUCKET')
host = os.getenv('HOST_SQL_ALCHEMY')
port = os.getenv('PORT_SQL_ALCHEMY')
database = os.getenv('DATABASE_SQL_ALCHEMY')
user = os.getenv('USER_SQL_ALCHEMY')
password = os.getenv('PASSWORD_SQL_ALCHEMY')

In [ ]:
# Read json file with the cities' list to visit
df_source = pd.read_json("best_cities_france.json")
df_source = df_source.rename(columns={0: "city"})
df_source.head()

In [ ]:
# Create a copy to avoid modifying the original dataframe
df = df_source.copy()

In [ ]:
# Define headers for APIs' calls
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}

# Geolocation Part

In [ ]:
for index, row in df.iterrows():
    # API Call to Nominatim
    res = requests.get(f"https://nominatim.openstreetmap.org/search?q={row['city']},France&format=json", headers=headers)
    city = res.json()[0]
    df.loc[index, "lat"] = city["lat"]
    df.loc[index, "lon"] = city["lon"]
    time.sleep(1)
df.head()

# Store result to reuse it later and avoid recalling API
df.to_csv("cities_with_geoposition.csv", index=False)

In [ ]:
# To execute if Nominatim API already stored
df = pd.read_csv("cities_with_geoposition.csv")
df.head()

# Weather Forecast Part

In [ ]:
list_weather_data = []

for index, row in df.iterrows():
    # API Call to Openweathermap
    res_weather = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={row['lat']}&lon={row['lon']}&units=metric&exclude=current,minutely,hourly,alerts&appid={weather_api_secret}", headers=headers)
    res_weather_json = res_weather.json()
    
    # Store current weather and forecast
    for res in res_weather_json['list']:
        weather_entry = {
            "city": row['city'],
            "lat": row['lat'],
            "lon": row['lon'],
            "date": datetime.datetime.fromtimestamp(res['dt']).strftime('%Y-%m-%d'),
            "hour": datetime.datetime.fromtimestamp(res['dt']).strftime('%H:%M'),
            "temp": res['main']['temp'],
            "prob_rain": res['pop'],
            "volume_rain": res.get('rain', {}).get('3h', 0),
            "wind_speed": res['wind']['speed'],
            "perc_cloud": res['clouds']['all']
        }
        list_weather_data.append(weather_entry)
    time.sleep(1)

df_weather = pd.DataFrame(list_weather_data)
print(df_weather.head())

# Store result to reuse it later and avoid recalling API
df_weather.to_csv("weather_forecast.csv", index=False)

In [ ]:
# To execute if Openweathermap API already stored
df_weather = pd.read_csv("weather_forecast.csv")
df_weather.head()

In [ ]:
# Rescaling 'prob_rain' to a 100-point scale & 'wind_speed' to km/h
df_weather['prob_rain'] = df_weather['prob_rain'] * 100
df_weather['wind_speed'] = df_weather['wind_speed'] * 3.6
df_weather.head()

In [ ]:
# Aggregate data for Weather Score calculation
df_weather_groupby = df_weather.groupby(['city', 'lat', 'lon', 'date']).agg({'temp': ['mean', 'min', 'max'], 'prob_rain': 'max', 'volume_rain': {'mean', 'max', 'sum'}, 'wind_speed': 'max', 'perc_cloud': 'mean'}).reset_index()
df_weather_groupby.head()

In [ ]:
# --- SCORING METHODOLOGY ---
# 1. NORMALIZATION (0-100):
#    Goal : 100 = Perfect, 0 = Horrible
#    Transform raw metrics into satisfaction scores.
#    - Multipliers (*4, *5) adjust sensitivity: 'Hard Penalty' vs 'Soft Penalty'.
#    - Example: 1°C gap = -4pts (Strict).

# Temperature (Target / Reference: 25°C).
# Minus 4 pts per degrees gap.
df_weather_groupby['score_temp'] = 100 - (abs(df_weather_groupby[('temp', 'max')] - 25) * 4)
df_weather_groupby['score_temp'] = df_weather_groupby['score_temp'].clip(lower=0)

# Rain Probability : 0% = 100 pts. 100% = 0 pts.
df_weather_groupby['score_rain_prob'] = 100 - (df_weather_groupby[('prob_rain', 'max')])

# Rain Volume : 0mm = 100 pts.
# Minut 5 pts per mm.
df_weather_groupby['score_rain_vol'] = 100 - (df_weather_groupby[('volume_rain', 'sum')] * 5)
df_weather_groupby['score_rain_vol'] = df_weather_groupby['score_rain_vol'].clip(lower=0)

# Wind Speed (km/h) : 0 km/h = 100 pts.
df_weather_groupby['score_wind'] = 100 - df_weather_groupby[('wind_speed', 'max')]
df_weather_groupby['score_wind'] = df_weather_groupby['score_wind'].clip(lower=0)

# Cloudness : 0% = 100 pts.
df_weather_groupby['score_cloud'] = 100 - df_weather_groupby[('perc_cloud', 'mean')]

# 2. Weighted Final Score
# Weights importance : Temp(30%), PluieProba(20%), PluieVol(30%), Vent(10%), Nuages(10%)
df_weather_groupby['total_score'] = (
    df_weather_groupby['score_temp'] * 0.3 +
    df_weather_groupby['score_rain_prob'] * 0.2 +
    df_weather_groupby['score_rain_vol'] * 0.3 +
    df_weather_groupby['score_wind'] * 0.1 +
    df_weather_groupby['score_cloud'] * 0.1
)

# 3. TOP 5 cities (Average over the period)
top_cities = df_weather_groupby.groupby(['city', 'lat', 'lon'])['total_score'].mean().sort_values(ascending=False)
print("--- FINAL RANKING ---")
print(top_cities.head(10))

# Store result to reuse it later and avoid run scoring again
df_weather_groupby.to_csv("weather_forecast_with_score.csv", index=False)

In [ ]:
# Transform Series in DataFrame for visualization
df_top_cities = pd.DataFrame(top_cities.reset_index())
df_top_cities_top_10 = df_top_cities.loc[0:9,:]
df_top_cities_top_10

In [ ]:
# Vsiualize the top 10 cities based on weather score for the next 5 days
fig = px.scatter_mapbox(
    df_top_cities_top_10, 
    lat="lat", 
    lon="lon",
    color="total_score",
    size="total_score", 
    color_continuous_scale=px.colors.cyclical.IceFire,
    size_max=15,
    zoom=4, 
    center={"lat": 46.2276, "lon": 2.2137}, # Center in France
    mapbox_style="carto-positron", 
    hover_name="city",
    title="Top 10 Destinations in France"
)

fig.show()

# Booking Scraping Part

In [ ]:
try:
    os.remove('hotels.json')
except OSError:
    pass

!cd booking_scraper_project && scrapy crawl booking_spider -O ../hotels.json

# Hotels' Visualization

In [ ]:
# Read hotels' data scraped
df_hotels = pd.read_json("hotels.json")

In [ ]:
# Clean & format data
df_hotels = df_hotels.dropna()

df_hotels['score'] = df_hotels['score'].str.replace(',', '.').astype(float)
df_hotels = df_hotels.sort_values(by=["score"], ascending=False)

df_hotels.head()

In [ ]:
# Visualization Top X hotels in France
fig = px.scatter_mapbox(
    df_hotels, 
    lat="lat", 
    lon="lng",
    color="score",
    size="score", 
    color_continuous_scale=px.colors.cyclical.IceFire,
    size_max=15,
    zoom=4, 
    center={"lat": 46.2276, "lon": 2.2137}, # Centré sur la France
    mapbox_style="carto-positron", 
    hover_name="name",
    title="Top 20 Hotels in France"
)

fig.show()

# Data Formating For Upload

In [ ]:
# Generate unique city's ID
df_top_cities['id'] = df_top_cities.apply(lambda x: uuid.uuid4(), axis=1)
print(df_top_cities.head())
df_top_cities.rename(columns={'city': 'name'}).to_csv("final_output/villes_table.csv", index=False)

In [ ]:
def merge_dataframes(df_to_merge, df_city, output_name):
    # 3. Merge
    df_to_merge = df_to_merge.merge(df_city, on='city')

    # Clean and renamed columns
    df_to_merge = df_to_merge.drop(['city'], axis=1)
    df_to_merge = df_to_merge.rename(columns={"id": "city_id"})

    # Generate unique weather's ID
    df_to_merge['id'] = df_to_merge.apply(lambda x: uuid.uuid4(), axis=1)

    df_to_merge.to_csv(f"final_output/{output_name}.csv", index=False)

    return df_to_merge

In [ ]:
df_top_cities_light = df_top_cities[['id', 'city']]
# 1. Reset l'index pour sortir 'city' et 'date'
df_weather_flat = df_weather_groupby.reset_index()

# 2. Flatten column names (if MultiIndex)
# Take level 0 (e.g. “temp”) unless it is empty, otherwise keep level 1.
df_weather_flat.columns = ['_'.join(c).strip('_') for c in df_weather_flat.columns.to_flat_index()]

# Columns to keep for merging
columns_weather_merging = ['city', 'date', 'temp_mean', 'temp_min',
'temp_max', 'prob_rain_max', 'volume_rain_mean', 'volume_rain_max',
'volume_rain_sum', 'wind_speed_max', 'perc_cloud_mean', 'score_temp',
'score_rain_prob', 'score_rain_vol', 'score_wind', 'score_cloud',
'total_score']

df_weather_flat_light = df_weather_flat[columns_weather_merging]

# 3. Merge and store
df_weather_final = merge_dataframes(df_weather_flat_light, df_top_cities_light, "weather_table")
df_hotel_final = merge_dataframes(df_hotels, df_top_cities_light, "hotels_table")

# Amazon S3 Part

In [ ]:
import boto3

In [ ]:
session = boto3.Session(aws_access_key_id=f"{aws_access_key_id}", aws_secret_access_key=f"{aws_secret_access_key}")

In [ ]:
s3 = session.resource("s3")

bucket = s3.create_bucket(Bucket=f"{bucket_name}")

# To review for a loop

In [ ]:
files = [f for f in os.listdir("final_output") if f.endswith('.csv')]

for file in files:
    s3.Bucket(bucket_name).upload_file(f"final_output/{file}", f"{file}.csv")

In [ ]:
# Import sqlalchemy
from sqlalchemy import create_engine, text

# Create engine will create a connection between DB and python
conf ={
    'host': f"{host}",
    'port': f"{port}",
    'database': f"{database}",
    'user': f"{user}",
    'password': f"{password}"
}
#engine = create_engine("postgresql+psycopg2://{user}:{password}@{host}/{database}".format(**conf))
engine = create_engine(f"postgresql+psycopg2://{conf['user']}:{conf['password']}@{conf['host']}/{conf['database']}", echo=True)

In [ ]:
from sqlalchemy import Table, Column, Integer, String, MetaData, ForeignKey, Float, Date

meta = MetaData()

# Define & create table "cities"
cities = Table(
    'cities', meta,
    Column('id', String, primary_key = True),
    Column('name', String),
    Column('lat', Float),
    Column('lon', Float),
    Column('total_score', Float)
)
meta.create_all(engine)

# Upload DataFrame to DB
df_top_cities.rename(columns={'city': 'name'}).to_sql('cities', engine, if_exists='append', index=False)


In [ ]:
# Define & create table "weather"
weather = Table(
    'weather', meta,
    Column('id', String, primary_key = True),
    Column('date', Date),
    Column('temp_mean', Float),
    Column('temp_min', Float),
    Column('temp_max', Float),
    Column('prob_rain_max', Float),
    Column('volume_rain_mean', Float),
    Column('volume_rain_max', Float),
    Column('volume_rain_sum', Float),
    Column('wind_speed_max', Float),
    Column('perc_cloud_mean', Float),
    Column('score_temp', Float),
    Column('score_rain_prob', Float),
    Column('score_rain_vol', Float),
    Column('score_wind', Float),
    Column('score_cloud', Float),
    Column('total_score', Float),
    Column('city_id', String, ForeignKey('cities.id'))
)

meta.create_all(engine)

# Format 'date' column & upload DataFrame to DB
df_weather_final['date'] = pd.to_datetime(df_weather_final['date'], dayfirst=True)
df_weather_final.to_sql('weather', engine, if_exists='append', index=False)


In [ ]:
# Define & create table "hotels"
hotels = Table(
    'hotels', meta,
    Column('id', String, primary_key = True),
    Column('name', String),
    Column('url', String),
    Column('score', Float),
    Column('lat', Float),
    Column('lng', Float),
    Column('description', String),
    Column('city_id', ForeignKey('cities.id'))
)

meta.create_all(engine)

# Upload DataFrame to DB
df_hotel_final.to_sql('hotels', engine, if_exists='append', index=False)
